# Part 3.2 Deploy

After this notebook you can ship the agent to LangSmith with one file and one command, and call it
from LangGraph code.

In [ ]:
from kea.config import setup

setup("kea-rebooking-local")

## See it

One file tells LangSmith what to serve. One command ships it. The first run creates the
deployment. Later runs ship a new revision.

What `mise run deploy` prints (about 2 minutes once the image is built):

```text
1. Looking up deployment 'kea-rebooking'
   No deployment found. Will create after build.
2. Creating deployment 'kea-rebooking'
3. Building image
4. Requesting push token
5. Logging into us-central1-docker.pkg.dev
6. Pushing image .../kea-rebooking:latest
7. Updating deployment
QUEUED... AWAITING_DEPLOY... DEPLOYING... (1m 29s)
   Deployment successful!
   URL: https://kea-rebooking-<id>.us.langgraph.app
```

<img src="../../assets/screenshots/p3-02-revisions.png" alt="LangSmith → Deployments → kea-rebooking → Revisions, with the latest revision Deployed." width="900">

## Build it

`langgraph.json` lists the graphs to serve and where the code lives.

In [ ]:
from kea.config import REPO_ROOT

print((REPO_ROOT / "langgraph.json").read_text())

`mise run deploy` runs `langgraph deploy --name kea-rebooking --deployment-type dev` with the API
key from `.env`. The cell below is commented out so it is not re-run by accident.

In [ ]:
# !cd ../.. && mise run deploy
# Same thing without mise:
# !cd ../.. && uv run langgraph deploy --name kea-rebooking --deployment-type dev

Once the deployment is Ready, call it from LangGraph code. `RemoteGraph` wraps a deployment as a
graph, so another graph can use it as a node or a subgraph.

In [ ]:
# needs: deployment
import os

from langgraph.pregel.remote import RemoteGraph

DEPLOYMENT_URL = os.environ.get("KEA_DEPLOYMENT_URL")
if DEPLOYMENT_URL:
    remote = RemoteGraph("rebooking", url=DEPLOYMENT_URL, api_key=os.environ["LANGSMITH_API_KEY"])
    status = remote.invoke(
        {"messages": [{"role": "user", "content": "Is KA 1182 DAL to DEN still operating?"}]}
    )
    last = status["messages"][-1]
    content = last["content"] if isinstance(last, dict) else last.content
    if isinstance(content, list):
        content = " ".join(block.get("text", "") for block in content)
    print(content)
else:
    print(
        "KEA_DEPLOYMENT_URL is not set, so the deployment cells are skipped. Add the deployment URL to .env, restart the kernel, and run again."
    )

## Under the hood

<img src="../../assets/slides/langgraph-fault-tolerance.jpg" width="720">

- **What you deployed is an Agent Server.** It hosts every graph in `langgraph.json` and adds
  **threads** (conversation state), **runs** (one execution on a thread), **crons** (runs on a
  schedule), and a **store** (memory across threads).
- **Durable execution.** The server checkpoints after every step, so an interrupt, a crash, or a
  redeploy resumes where it stopped. This is why `agent.py` passes no checkpointer or store: the
  platform provides both.
- **Where it runs.** LangSmith Cloud hosts it for you. The same server runs
  [self-hosted or in your own cloud (BYOC)](https://docs.langchain.com/langsmith/deploy-to-self-hosted-overview)
  when data must stay in your network. Same code, same API.
- **Deployment types.** `dev` is the legacy name for the small, cheap type. Newer CLIs default to
  `serverless`, which scales to zero, with `dedicated` for always-on capacity. The type is fixed
  when the deployment is created.
- **Tracing is automatic.** The platform creates a tracing project for the deployment and sets
  the tracing environment variables.
- **`RemoteGraph`** takes the graph id, the URL, and an API key. Docs:
  [Use a remote graph](https://docs.langchain.com/langsmith/use-remote-graph),
  [Deploy to Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).

## Recap

- `langgraph.json` plus `langgraph deploy` gives you an API with threads, durable state, and traces.
- The platform supplies the checkpointer and store.
- Cloud, self-hosted, and BYOC run the same server.

**Next:** [3.3 Online evaluators](03_online_evaluators.ipynb)